# Grover's algorithm: from linear algebra to quantum gates

This notebook implements Grover's algorithm in two ways. First, we use simple linear algebra to see what happens to the amplitudes. Then, we build the same idea using quantum gates in Qiskit!

The implementation is based on [Grover's original paper](https://arxiv.org/abs/quant-ph/9605043).

## Running the notebook

Install the dependencies in your notebook's Python environment:

```bash
python -m pip install numpy matplotlib qiskit qiskit-aer
```

Open the notebook in Jupyter or VS Code and run the cells from top to bottom. No IBM Quantum account is needed for the local simulation.

## 1. Grover's algorithm using simple linear algebra

Here, we work directly with the state vector and matrices. Let $n$ be the number of qubits and $N = 2^n$ the number of computational basis states. We assume one marked state.

### The algorithm

- **Input:** the number of qubits $n$ and an oracle for a function $f$, where $f(x)=1$ only for the marked state.
- **Output:** a final state vector $\lvert\psi_f\rangle$, whose measurement probabilities are $p(x)=|\langle x\vert\psi_f\rangle|^2$ — hopefully concentrated on the desired solution!

1. Start with a uniform superposition, prepared by applying a Hadamard gate to each qubit:

   $$
   \lvert s\rangle = H^{\otimes n}\lvert 0^n\rangle
   = \frac{1}{\sqrt{N}}\sum_{x=0}^{N-1}\lvert x\rangle.
   $$

2. Repeat the oracle and diffusion steps about $\frac{\pi}{4}\sqrt{N}$ times:

   - **Oracle:** flip the phase of the marked state and leave the other states unaltered:

     $$
     O_f\lvert x\rangle = (-1)^{f(x)}\lvert x\rangle.
     $$

   - **Diffusion:** apply inversion about the average amplitude:

     $$
     D = 2\lvert s\rangle\langle s\rvert - I.
     $$

3. Measure the final state to obtain a candidate solution. In this first implementation, we inspect the amplitudes after each iteration instead.

The state vector contains **probability amplitudes**. To obtain the probability mass function, we take their squared magnitudes: `np.abs(psi_vector)**2`. The oracle changes a phase; the diffusion step turns that phase difference into amplitude amplification.

We use **5 qubits**, mark $\lvert 00001\rangle$, and apply **4 Grover iterations**. The oracle is given the marked index explicitly so we can study the amplification. In a search problem, we would build the oracle from a condition that recognizes valid solutions.

This is a small, explicit simulation. The state vector stores $2^n$ amplitudes, and the dense matrices below store $4^n$ entries each. Grover's $\mathcal{O}(\sqrt{N})$ advantage concerns oracle queries on a quantum computer; this NumPy implementation does not demonstrate a classical runtime speedup. Next, let's express the same operations with gates.

In [132]:
import numpy as np
import matplotlib.pyplot as plt


def diffusion_transformation(R, H_matrix, num_qubits): # explicit matrices first; gates come next!
    '''
    D = H^(tensor n) R H^(tensor n), where R = 2|0^n><0^n| - I.
    This gives D_ii = -1 + 2/N and D_ij = 2/N for i != j, with N = 2^n.
    In other words, each amplitude a_i becomes 2 * mean(a) - a_i.
    '''
    H_kron_n = H_matrix

    for _ in range(1, num_qubits):
        H_kron_n = np.kron(H_kron_n, H_matrix)

    return(H_kron_n @ R @ H_kron_n)

def oracle(psi_vector, desired_state_idx): # a known marked state, just to see the amplification
    # Flip the marked amplitude's sign in place; its probability is unchanged by this step alone.
    psi_vector[desired_state_idx] = -1*psi_vector[desired_state_idx]
    return(psi_vector)


num_qubits= 5
desired_state_idx = 1 # |00001> for five qubits
# These are uniform amplitudes; each initial measurement probability is 1 / 2^n.
uniform_initial_distribution = 1/(np.sqrt(2**num_qubits))*np.ones(2**num_qubits)
H_matrix = 1/np.sqrt(2)*np.array([[1, 1], [1, -1]])
R = -1*np.eye(2**num_qubits) # R leaves |0^n> unchanged and flips every other basis state's phase
R[0,0]=1
D = diffusion_transformation(R, H_matrix, num_qubits)


grover_iteration = 4
psi_vector =  uniform_initial_distribution

for _ in range(grover_iteration):
    psi_vector = oracle(psi_vector, desired_state_idx)
    psi_vector = D.dot(psi_vector)
    print(psi_vector) # amplitudes after one oracle + diffusion step

[0.15467961 0.508233   0.15467961 0.15467961 0.15467961 0.15467961
 0.15467961 0.15467961 0.15467961 0.15467961 0.15467961 0.15467961
 0.15467961 0.15467961 0.15467961 0.15467961 0.15467961 0.15467961
 0.15467961 0.15467961 0.15467961 0.15467961 0.15467961 0.15467961
 0.15467961 0.15467961 0.15467961 0.15467961 0.15467961 0.15467961
 0.15467961 0.15467961]
[0.11324757 0.77616018 0.11324757 0.11324757 0.11324757 0.11324757
 0.11324757 0.11324757 0.11324757 0.11324757 0.11324757 0.11324757
 0.11324757 0.11324757 0.11324757 0.11324757 0.11324757 0.11324757
 0.11324757 0.11324757 0.11324757 0.11324757 0.11324757 0.11324757
 0.11324757 0.11324757 0.11324757 0.11324757 0.11324757 0.11324757
 0.11324757 0.11324757]
[0.05765959 0.94706733 0.05765959 0.05765959 0.05765959 0.05765959
 0.05765959 0.05765959 0.05765959 0.05765959 0.05765959 0.05765959
 0.05765959 0.05765959 0.05765959 0.05765959 0.05765959 0.05765959
 0.05765959 0.05765959 0.05765959 0.05765959 0.05765959 0.05765959
 0.05765959 0.

## 2. Grover's algorithm using Qiskit

Now, let's build the oracle and diffusion operator using quantum gates. This example uses **10 qubits**, one marked state, and **1,000 measurement shots** on Qiskit's local `AerSimulator`.

We no longer construct the full matrices ourselves. The circuit describes operations on qubits, including multi-qubit controlled gates. The simulator still runs classically, and a full statevector simulation still needs $2^n$ amplitudes. **This notebook runs an ideal simulation; it does not execute on IBM hardware.**

Remember that Qiskit writes bitstrings as $q_{n-1}\ldots q_1q_0$, with qubit `0` on the right. We reverse `marked_state` when mapping its bits to qubit indices. See [Qiskit's bit-ordering guide](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering).

### How many iterations?

For one marked state, let $\theta=\arcsin(1/\sqrt{N})$. After $k$ Grover iterations, the ideal success probability is

$$
P_k = \sin^2\left((2k+1)\theta\right).
$$

The code rounds down $\frac{\pi}{4\theta}-\frac{1}{2}$, giving **24 iterations** for $N=1024$, near the first probability peak. More iterations are not always better — we can rotate past the marked state!

In [5]:
# Let's see the same idea with gates! This example uses 10 qubits on a local simulator.
from qiskit import *
from qiskit.quantum_info import *
from math import pi
from qiskit_aer import AerSimulator
from qiskit import transpile
import numpy as np

rng = np.random.default_rng()


def oracle(num_qubits, marked_state):
    # marked_state = |00101101...1>, written as |q_(n-1)...q_1q_0>
    '''
    Flip the phase of the marked state using the following steps:
        - Apply X to each qubit whose marked bit is 0, mapping the marked state to all ones.
        - Use H-MCX-H on the target qubit to flip the phase of |111...111>.
        - Undo the X gates to return to the original basis.

    Inputs:
        - num_qubits: number of qubits (at least 2 in this example).
        - marked_state: a binary string of length num_qubits, e.g. "100".

    Output:
        - qc: the existing circuit, modified in place and returned.
          Here, qc is defined below and used by both circuit-building functions.
    '''

    # Remember: Qiskit puts the most significant bit on the left, so we reverse the string.
    zeros_indices = [i for i, bit in enumerate(reversed(marked_state)) if bit=="0"]

    # Map the marked state to |111...111>.
    for idx in zeros_indices: qc.x(idx)

    # H-MCX-H implements MCZ. We choose qubit n-1 as the target.
    qc.h(num_qubits-1)
    qc.mcx(list(range(num_qubits-1)), num_qubits-1)
    qc.h(num_qubits-1)

    # Undo the X gates.
    for idx in zeros_indices: qc.x(idx)

    return(qc)


def Grover_diffusion_operator(num_qubits):
    '''
    Apply D = H^(tensor n) (2|0^n><0^n| - I) H^(tensor n):
        - Apply Hadamard gates to all qubits.
        - Apply X gates, then H-MCX-H, then undo the X gates.
          This flips the phase of |0^n>, giving I - 2|0^n><0^n|.
        - Add a global phase of pi to obtain 2|0^n><0^n| - I.
        - Apply Hadamard gates to all qubits again.

    Input:
        - num_qubits: number of qubits (at least 2 in this example).

    Output:
        - qc: the existing circuit, modified in place and returned.
    '''

    # Move from the uniform-superposition reflection to the |0^n> reflection.
    for i in range(num_qubits): qc.h(i)

    # X-MCZ-X flips the phase of |0^n>.
    for i in range(num_qubits): qc.x(i)
    # MCZ via H-MCX-H on the target qubit.
    qc.h(num_qubits-1)
    qc.mcx(list(range(num_qubits-1)), num_qubits-1)
    qc.h(num_qubits-1)
    for i in range(num_qubits): qc.x(i)

    # Match 2|0^n><0^n| - I exactly; this global phase does not affect probabilities.
    qc.global_phase += pi

    # Return to the uniform-superposition reflection.
    for i in range(num_qubits): qc.h(i)

    return(qc)


num_qubits = 10
qc = QuantumCircuit(num_qubits)
marked_state = bin(rng.integers(0, 2**num_qubits-1))[2:]

# Pad the bitstring so that it has one bit per qubit.
if len(marked_state) != num_qubits:
    leading_zeros = "0"*(num_qubits-len(marked_state))
    marked_state = leading_zeros  + marked_state


print(f"This is the desired state: {marked_state}")

# Round down the iteration count at the first theoretical peak: 24 for 10 qubits.
num_iterations = int(((np.pi/2.0)/(np.arcsin(1/np.sqrt(2**num_qubits)))-1)/2)

# Prepare the uniform superposition.
for i in range(num_qubits): qc.h(i)


for _ in range(num_iterations):
    qc = oracle(num_qubits, marked_state)
    qc = Grover_diffusion_operator(num_qubits)


qc.measure_all()

# Run an ideal local simulation; no hardware backend or noise model is used here.
simulator = AerSimulator()
compiled_qc = transpile(qc, simulator)

result = simulator.run(compiled_qc, shots=1000).result()
counts = result.get_counts()

print(counts)


# Optional: draw the circuit. The full 10-qubit circuit is quite large!
# qc.draw(output='mpl', fold=-1)

# Optional: inspect the state before measurement (keep num_qubits small).
# psi = Statevector.from_instruction(qc.remove_final_measurements(inplace=False))
# display(psi.draw("latex"))

This is the desired state: 0010111001
{'1101110011': 1, '0010111001': 999}


## Reading the results

In the linear algebra example, the marked state's probability after 4 iterations is about **99.92%**, obtained from `np.abs(psi_vector[desired_state_idx])**2`.

The saved Qiskit run gives **999 out of 1,000 shots** for the marked state `0010111001`, consistent with the ideal probability of about **99.85%**. Each run chooses a new marked state, and the counts fluctuate. A few other outcomes are expected even without noise.

### Next step: real hardware

I would like to try a small version on IBM hardware and compare it with the ideal simulation. Of course, we expect noise and circuit depth to matter — let's see it for ourselves! Scaling up requires more than having enough qubits: the number of Grover iterations and the cost of the controlled gates matter too. IBM's [Grover tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/grovers-algorithm) discusses these hardware limitations.